# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Baseline X0: Fast Inverted-Index BM25 + TF-IDF Lexical Retriever & Local Evaluator

* **Task**: Prior Case Retrieval (PCR) — For each legal query case, retrieve and rank exactly **7 unique candidate case IDs** from its split pool.
* **Primary Target Metric**: **Micro-F1@7** (Secondary: Micro-Precision@7, Micro-Recall@7).
* **Role in Phase 7 Strategy**:
  1. **Immediate Score Insurance**: Strong lexical baseline that can be submitted to CodaBench in minutes.
  2. **Lexical Foundation**: In legal case retrieval, statutes and precedent citations are highly discriminative. BM25 provides the lexical signals that feed the subsequent hybrid ranker.
  3. **Local Benchmark**: Exact evaluation module used across all 6 deep models for consistent model selection.
* **Core Components**:
  - Sparse inverted-index BM25Okapi ($k_1=1.2, b=0.75$) with sub-second per-query latency.
  - Sublinear-scaled TF-IDF 1–2-gram cosine similarity.
  - Reciprocal Rank Fusion (RRF, $k=60$) & score-level convex blending.
  - Strict CodaBench submission packaging with verification assertions.


In [ ]:
# 1. Environment Dependencies & Hugging Face Authentication
import os
import sys
import gc
import json
import math
import re
import time
import zipfile
import shutil
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Set, Any, Optional

import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

# Hugging Face Hub APIs
from huggingface_hub import HfApi, login, hf_hub_download

# Authenticate via Kaggle Secrets or Environment
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("✓ Hugging Face authentication successful.")
else:
    print("ℹ No HF_TOKEN supplied. Public access or local datasets will be used.")


In [ ]:
# 2. Execution Profile & Path Configuration
PROFILE = "CORE"  # Options: "CORE" (Full Evaluation & Submission) or "SMOKE" (Rapid Sanity Check)

PROFILES = {
    "SMOKE": {
        "max_train_queries": 30,
        "max_dev_queries": 20,
        "max_test_queries": 20,
        "max_candidates": 200,
        "description": "Rapid Sanity Check (~15 seconds)"
    },
    "CORE": {
        "max_train_queries": None,
        "max_dev_queries": None,
        "max_test_queries": None,
        "max_candidates": None,
        "description": "Full Production Run (~2 minutes)"
    }
}

CFG = PROFILES[PROFILE]
print(f"=== Active Profile: {PROFILE} ({CFG['description']}) ===")

# Repositories & Working Directory
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_DEST_REPO = "sarweshbarnwal/CS781-Legal-Models"
EXPERIMENT_NAME = "baseline_x0_lexical_bm25_tfidf"
HUB_SUBFOLDER = f"Phase_7/{EXPERIMENT_NAME}"

OUTPUT_DIR = "/kaggle/working/phase7_x0" if os.path.exists("/kaggle") else "./phase7_x0"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
METRICS_PATH = os.path.join(OUTPUT_DIR, "eval_metrics.json")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Artifact output path: {OUTPUT_DIR}")


In [ ]:
# 3. Data Ingestion & Dataset Diagnostics
print("--- Loading Phase 7 Dataset Splits ---")

def load_pcr_json(filename: str):
    # Check common Kaggle dataset mount locations first
    local_candidates = [
        os.path.join("/kaggle/input/cs781-phase7-data", "Phase_7_PCR", filename),
        os.path.join("/kaggle/input/cs781-phase7", "Phase_7_PCR", filename),
        os.path.join("/kaggle/input/cs781-phase7-data", filename),
        os.path.join("./Phase_7_PCR", filename),
        os.path.join("./phase7_data", filename)
    ]
    for p in local_candidates:
        if os.path.exists(p):
            print(f"  Loading local file: {p}")
            with open(p, "r", encoding="utf-8") as f:
                return json.load(f)
    
    # Download from Hugging Face Hub if not local
    print(f"  Downloading Phase_7_PCR/{filename} from {HF_DATASET_REPO}...")
    fpath = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=f"Phase_7_PCR/{filename}",
        repo_type="dataset",
        token=HF_TOKEN
    )
    with open(fpath, "r", encoding="utf-8") as f:
        return json.load(f)

raw_train_queries = load_pcr_json("Train/train.json")
raw_train_cands   = load_pcr_json("Train/train_candidates.json")
raw_dev_queries   = load_pcr_json("Dev/dev.json")
raw_dev_cands     = load_pcr_json("Dev/dev_candidates.json")
raw_test_queries  = load_pcr_json("Test/test.json")
raw_test_cands    = load_pcr_json("Test/test_candidates.json")

def slice_dict(d: dict, limit: Optional[int]):
    if limit is None:
        return d
    return {k: d[k] for k in list(d.keys())[:limit]}

train_queries = slice_dict(raw_train_queries, CFG["max_train_queries"])
train_cands   = slice_dict(raw_train_cands, CFG["max_candidates"])
dev_queries   = slice_dict(raw_dev_queries, CFG["max_dev_queries"])
dev_cands     = slice_dict(raw_dev_cands, CFG["max_candidates"])
test_queries  = slice_dict(raw_test_queries, CFG["max_test_queries"])
test_cands    = slice_dict(raw_test_cands, CFG["max_candidates"])

print("\n=== Ingested Dataset Summary ===")
print(f"  Train: {len(train_queries):,} queries | {len(train_cands):,} candidate pool")
print(f"  Dev:   {len(dev_queries):,} queries | {len(dev_cands):,} candidate pool")
print(f"  Test:  {len(test_queries):,} queries | {len(test_cands):,} candidate pool")

# Diagnostics on Gold Set Distribution
dev_gold_sizes = [len(q.get("relevant_candidates", [])) for q in dev_queries.values()]
print("\nDev Gold Set Size Distribution:")
print(f"  Min: {min(dev_gold_sizes)} | Median: {int(np.median(dev_gold_sizes))} | Max: {max(dev_gold_sizes)} | Mean: {np.mean(dev_gold_sizes):.2f}")
print(f"  Queries with >= 7 gold candidates: {sum(s >= 7 for s in dev_gold_sizes)} / {len(dev_gold_sizes)} ({sum(s >= 7 for s in dev_gold_sizes)/len(dev_gold_sizes)*100:.1f}%)")


In [ ]:
# 4. High-Performance Sparse Inverted-Index BM25 Engine
TOKEN_RE = re.compile(r"\b[a-zA-Z0-9_]{2,}\b")

def tokenize_legal_text(text_obj) -> List[str]:
    if isinstance(text_obj, list):
        full_text = " ".join(text_obj)
    else:
        full_text = str(text_obj)
    return TOKEN_RE.findall(full_text.lower())

class FastBM25:
    """
    Sparse inverted-index BM25Okapi engine.
    Complexity: O(query_terms * avg_postings_length).
    Scores thousands of candidates in milliseconds.
    """
    def __init__(self, corpus_dict: Dict[str, dict], k1: float = 1.2, b: float = 0.75):
        self.doc_ids = [str(k) for k in corpus_dict.keys()]
        self.doc_id_to_idx = {did: idx for idx, did in enumerate(self.doc_ids)}
        self.N = len(self.doc_ids)
        self.k1 = k1
        self.b = b
        
        # Build inverted index
        self.postings = defaultdict(list)  # term -> list of (doc_idx, tf)
        self.doc_len = np.zeros(self.N, dtype=np.float32)
        
        t0 = time.time()
        for idx, did in enumerate(self.doc_ids):
            tokens = tokenize_legal_text(corpus_dict[did].get("text", []))
            self.doc_len[idx] = len(tokens)
            tf_counts = Counter(tokens)
            for term, count in tf_counts.items():
                self.postings[term].append((idx, count))
                
        self.avg_doc_len = float(np.mean(self.doc_len)) if self.N > 0 else 1.0
        
        # Compute IDF for every term in vocabulary
        self.idf = {}
        for term, plist in self.postings.items():
            df = len(plist)
            self.idf[term] = math.log(1.0 + (self.N - df + 0.5) / (df + 0.5))
            
        # Precompute length normalization denominators
        self.len_norm = (1.0 - self.b) + self.b * (self.doc_len / max(1e-6, self.avg_doc_len))
        print(f"✓ BM25 index built in {time.time()-t0:.2f}s. Vocab: {len(self.postings):,} terms | Avg Doc Len: {self.avg_doc_len:.1f} tokens.")

    def score_query(self, query_tokens: List[str], exclude_id: Optional[str] = None) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        q_counts = Counter(query_tokens)
        
        for term, q_tf in q_counts.items():
            if term not in self.postings:
                continue
            idf_val = self.idf[term]
            p_list = self.postings[term]
            
            for doc_idx, doc_tf in p_list:
                denom = doc_tf + self.k1 * self.len_norm[doc_idx]
                scores[doc_idx] += idf_val * (doc_tf * (self.k1 + 1.0)) / denom
                
        if exclude_id is not None and exclude_id in self.doc_id_to_idx:
            scores[self.doc_id_to_idx[exclude_id]] = -1e9
            
        return scores

    def rank_query(self, query_tokens: List[str], top_k: int = 7, exclude_id: Optional[str] = None) -> List[str]:
        scores = self.score_query(query_tokens, exclude_id=exclude_id)
        if top_k >= self.N:
            top_indices = np.argsort(scores)[::-1]
        else:
            top_indices = np.argpartition(scores, -top_k)[-top_k:]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        return [self.doc_ids[i] for i in top_indices]


In [ ]:
# 5. TF-IDF 1–2-Gram Vectorizer & Cosine Similarity Engine
class FastTFIDF:
    """
    Sublinear-scaled TF-IDF (1, 2)-gram Cosine Similarity Engine.
    Captures multi-word legal collocations ('habeas corpus', 'penal code', 'reasonable doubt').
    """
    def __init__(self, corpus_dict: Dict[str, dict], min_df: int = 2, max_df: float = 0.95):
        self.doc_ids = [str(k) for k in corpus_dict.keys()]
        self.doc_id_to_idx = {did: idx for idx, did in enumerate(self.doc_ids)}
        self.N = len(self.doc_ids)
        
        t0 = time.time()
        corpus_texts = [
            " ".join(corpus_dict[did].get("text", [])) if isinstance(corpus_dict[did].get("text", []), list)
            else str(corpus_dict[did].get("text", ""))
            for did in self.doc_ids
        ]
        
        self.vectorizer = TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=min_df,
            max_df=max_df,
            sublinear_tf=True,
            token_pattern=r"\b[a-zA-Z0-9_]{2,}\b"
        )
        self.D_tfidf = self.vectorizer.fit_transform(corpus_texts)  # Sparse matrix [N, V]
        print(f"✓ TF-IDF fitted in {time.time()-t0:.2f}s. Vocabulary size: {self.D_tfidf.shape[1]:,} features.")

    def score_query(self, query_text: str, exclude_id: Optional[str] = None) -> np.ndarray:
        q_vec = self.vectorizer.transform([query_text])  # [1, V]
        scores = (self.D_tfidf @ q_vec.T).toarray().flatten()  # Cosine similarities
        if exclude_id is not None and exclude_id in self.doc_id_to_idx:
            scores[self.doc_id_to_idx[exclude_id]] = -1e9
        return scores

    def rank_query(self, query_text: str, top_k: int = 7, exclude_id: Optional[str] = None) -> List[str]:
        scores = self.score_query(query_text, exclude_id=exclude_id)
        if top_k >= self.N:
            top_indices = np.argsort(scores)[::-1]
        else:
            top_indices = np.argpartition(scores, -top_k)[-top_k:]
            top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
        return [self.doc_ids[i] for i in top_indices]


In [ ]:
# 6. Official Local Evaluation Metric: Micro-F1@7
def evaluate_micro_f1_at_k(
    predictions: Dict[str, List[str]], 
    gold_records: Dict[str, dict], 
    k: int = 7
) -> Tuple[float, float, float]:
    """
    Computes official Micro-F1@k, Micro-Precision@k, and Micro-Recall@k.
    Matches CodaBench competition evaluation exactly:
      TP = sum_q |top_k_q ∩ gold_q|
      P  = TP / (k * total_queries)
      R  = TP / sum_q |gold_q|
      F1 = 2 * P * R / (P + R)
    """
    eval_qids = [q for q in gold_records if q in predictions]
    if not eval_qids:
        return 0.0, 0.0, 0.0
        
    total_tp = 0
    total_gold = 0
    
    for qid in eval_qids:
        preds = predictions[qid][:k]
        golds = set(gold_records[qid].get("relevant_candidates", []))
        total_tp += len(set(preds) & golds)
        total_gold += len(golds)
        
    total_retrieved = k * len(eval_qids)
    
    prec = total_tp / max(1, total_retrieved)
    rec = total_tp / max(1, total_gold)
    f1 = (2.0 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    
    return f1, prec, rec

print("✓ Micro-F1@7 evaluation module compiled.")


In [ ]:
# 7. Dev Split Evaluation & Fusion Optimization
print("=== Building Dev Split Indexers ===")
dev_bm25 = FastBM25(dev_cands, k1=1.2, b=0.75)
dev_tfidf = FastTFIDF(dev_cands, min_df=2, max_df=0.95)

print("\nRunning Evaluation on Dev Set Queries...")
bm25_dev_preds = {}
tfidf_dev_preds = {}
hybrid_rrf_preds = {}
hybrid_blended_preds = {}

# Precompute raw scores for fast tuning
bm25_dev_scores = []
tfidf_dev_scores = []
dev_qids = list(dev_queries.keys())

for qid in dev_qids:
    q_data = dev_queries[qid]
    q_text = " ".join(q_data.get("text", [])) if isinstance(q_data.get("text", []), list) else str(q_data.get("text", ""))
    q_tokens = tokenize_legal_text(q_text)
    
    s_bm25 = dev_bm25.score_query(q_tokens, exclude_id=str(qid))
    s_tfidf = dev_tfidf.score_query(q_text, exclude_id=str(qid))
    
    bm25_dev_scores.append(s_bm25)
    tfidf_dev_scores.append(s_tfidf)
    
    # 1. BM25 Top 7
    top7_bm25 = [dev_bm25.doc_ids[i] for i in np.argpartition(s_bm25, -7)[-7:][np.argsort(s_bm25[np.argpartition(s_bm25, -7)[-7:]])[::-1]]]
    bm25_dev_preds[qid] = top7_bm25
    
    # 2. TF-IDF Top 7
    top7_tfidf = [dev_tfidf.doc_ids[i] for i in np.argpartition(s_tfidf, -7)[-7:][np.argsort(s_tfidf[np.argpartition(s_tfidf, -7)[-7:]])[::-1]]]
    tfidf_dev_preds[qid] = top7_tfidf

f1_bm25, p_bm25, r_bm25 = evaluate_micro_f1_at_k(bm25_dev_preds, dev_queries, k=7)
f1_tfidf, p_tfidf, r_tfidf = evaluate_micro_f1_at_k(tfidf_dev_preds, dev_queries, k=7)

print(f"  [1] Standalone BM25:   Micro-F1@7 = {f1_bm25*100:.2f}% | Precision@7 = {p_bm25*100:.2f}% | Recall@7 = {r_bm25*100:.2f}%")
print(f"  [2] Standalone TF-IDF: Micro-F1@7 = {f1_tfidf*100:.2f}% | Precision@7 = {p_tfidf*100:.2f}% | Recall@7 = {r_tfidf*100:.2f}%")

# Grid search for optimal convex blending: alpha * BM25_norm + (1 - alpha) * TFIDF_norm
best_alpha = 0.6
best_blended_f1 = 0.0

for alpha in np.linspace(0.0, 1.0, 11):
    curr_preds = {}
    for idx, qid in enumerate(dev_qids):
        b_s = bm25_dev_scores[idx]
        t_s = tfidf_dev_scores[idx]
        
        # Min-max normalize valid entries
        b_min, b_max = b_s[b_s > -1e8].min(), b_s[b_s > -1e8].max()
        t_min, t_max = t_s[t_s > -1e8].min(), t_s[t_s > -1e8].max()
        
        b_norm = (b_s - b_min) / max(1e-6, b_max - b_min)
        t_norm = (t_s - t_min) / max(1e-6, t_max - t_min)
        b_norm[b_s <= -1e8] = -1e9
        t_norm[t_s <= -1e8] = -1e9
        
        comb = alpha * b_norm + (1.0 - alpha) * t_norm
        top7_comb = [dev_bm25.doc_ids[i] for i in np.argpartition(comb, -7)[-7:][np.argsort(comb[np.argpartition(comb, -7)[-7:]])[::-1]]]
        curr_preds[qid] = top7_comb
        
    cf1, cp, cr = evaluate_micro_f1_at_k(curr_preds, dev_queries, k=7)
    if cf1 > best_blended_f1:
        best_blended_f1 = cf1
        best_alpha = alpha

print(f"\n  [3] Optimal Blended (Alpha={best_alpha:.2f}): Micro-F1@7 = {best_blended_f1*100:.2f}%")

# Test Reciprocal Rank Fusion (RRF, k=60)
rrf_preds = {}
for idx, qid in enumerate(dev_qids):
    b_s = bm25_dev_scores[idx]
    t_s = tfidf_dev_scores[idx]
    
    b_ranks = np.zeros(len(b_s), dtype=np.float32)
    t_ranks = np.zeros(len(t_s), dtype=np.float32)
    
    b_order = np.argsort(b_s)[::-1]
    t_order = np.argsort(t_s)[::-1]
    
    b_ranks[b_order] = np.arange(len(b_s))
    t_ranks[t_order] = np.arange(len(t_s))
    
    rrf_score = (1.0 / (60.0 + b_ranks)) + (1.0 / (60.0 + t_ranks))
    rrf_score[b_s <= -1e8] = -1e9
    
    top7_rrf = [dev_bm25.doc_ids[i] for i in np.argpartition(rrf_score, -7)[-7:][np.argsort(rrf_score[np.argpartition(rrf_score, -7)[-7:]])[::-1]]]
    rrf_preds[qid] = top7_rrf

f1_rrf, p_rrf, r_rrf = evaluate_micro_f1_at_k(rrf_preds, dev_queries, k=7)
print(f"  [4] Reciprocal Rank Fusion (RRF-60): Micro-F1@7 = {f1_rrf*100:.2f}% | Precision@7 = {p_rrf*100:.2f}% | Recall@7 = {r_rrf*100:.2f}%")

winning_mode = "BLENDED" if best_blended_f1 >= f1_rrf else "RRF"
winning_f1 = max(best_blended_f1, f1_rrf, f1_bm25)
print(f"\n🏆 Winning Lexical Baseline Strategy: {winning_mode} with Micro-F1@7 = {winning_f1*100:.2f}%")


In [ ]:
# 8. Test Split Full Inference & Top-7 Ranking
print("=== Building Test Split Indexers ===")
test_bm25 = FastBM25(test_cands, k1=1.2, b=0.75)
test_tfidf = FastTFIDF(test_cands, min_df=2, max_df=0.95)

print(f"\nGenerating Top 7 predictions for {len(test_queries):,} test queries...")
predictions = {}
test_cand_ids = test_bm25.doc_ids

for qid, q_data in test_queries.items():
    q_str_id = str(qid)
    q_text = " ".join(q_data.get("text", [])) if isinstance(q_data.get("text", []), list) else str(q_data.get("text", ""))
    q_tokens = tokenize_legal_text(q_text)
    
    s_bm25 = test_bm25.score_query(q_tokens, exclude_id=q_str_id)
    s_tfidf = test_tfidf.score_query(q_text, exclude_id=q_str_id)
    
    if winning_mode == "BLENDED":
        b_min, b_max = s_bm25[s_bm25 > -1e8].min(), s_bm25[s_bm25 > -1e8].max()
        t_min, t_max = s_tfidf[s_tfidf > -1e8].min(), s_tfidf[s_tfidf > -1e8].max()
        
        b_norm = (s_bm25 - b_min) / max(1e-6, b_max - b_min)
        t_norm = (s_tfidf - t_min) / max(1e-6, t_max - t_min)
        b_norm[s_bm25 <= -1e8] = -1e9
        t_norm[s_tfidf <= -1e8] = -1e9
        
        final_scores = best_alpha * b_norm + (1.0 - best_alpha) * t_norm
    else:
        b_ranks = np.zeros(len(s_bm25), dtype=np.float32)
        t_ranks = np.zeros(len(s_tfidf), dtype=np.float32)
        b_order = np.argsort(s_bm25)[::-1]
        t_order = np.argsort(s_tfidf)[::-1]
        b_ranks[b_order] = np.arange(len(s_bm25))
        t_ranks[t_order] = np.arange(len(s_tfidf))
        
        final_scores = (1.0 / (60.0 + b_ranks)) + (1.0 / (60.0 + t_ranks))
        final_scores[s_bm25 <= -1e8] = -1e9
        
    top7_idx = np.argpartition(final_scores, -7)[-7:]
    top7_sorted = top7_idx[np.argsort(final_scores[top7_idx])[::-1]]
    top7_cands = [test_cand_ids[i] for i in top7_sorted]
    
    predictions[q_str_id] = top7_cands

print(f"✓ Generated ranked Top 7 predictions for all {len(predictions):,} test queries.")


In [ ]:
# 9. Strict CodaBench Verification Assertions
print("=== Strict CodaBench Submission Verification ===")

# Assertion 1: Exact Query Count
assert len(predictions) == len(test_queries), f"Mismatch: {len(predictions)} predictions vs {len(test_queries)} queries"

# Candidate pool set
test_cand_pool = set(str(k) for k in test_cands.keys())

for qid, cands in predictions.items():
    # Assertion 2: String IDs
    assert isinstance(qid, str), f"Query ID {qid} is not a string!"
    
    # Assertion 3: Exactly 7 Candidates
    assert len(cands) == 7, f"Query {qid} has {len(cands)} candidates, expected 7!"
    
    # Assertion 4: Uniqueness
    assert len(set(cands)) == 7, f"Query {qid} contains duplicate candidate IDs: {cands}"
    
    # Assertion 5: Self-Exclusion
    assert qid not in cands, f"Query {qid} contains itself in candidate predictions!"
    
    # Assertion 6: Membership in Test Candidate Pool
    for cid in cands:
        assert isinstance(cid, str), f"Candidate ID {cid} is not a string!"
        assert cid in test_cand_pool, f"Candidate {cid} not in official Test candidate pool!"

print("✓ All 6 strict CodaBench verification checks passed successfully!")


In [ ]:
# 10. Packaging submission.zip & Hugging Face Hub Synchronization
# Write prediction.json
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

file_kb = os.path.getsize(PREDICTION_PATH) / 1024
print(f"✓ Saved {PREDICTION_PATH} ({file_kb:.1f} KB)")

# Package submission.zip with prediction.json directly at the root
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

zip_kb = os.path.getsize(SUBMISSION_ZIP) / 1024
print(f"✓ Packaged {SUBMISSION_ZIP} ({zip_kb:.1f} KB)")

# Verify ZIP integrity
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zipf:
    contents = zipf.namelist()
    assert "prediction.json" in contents, "FATAL: prediction.json not at root of submission.zip!"
    assert len(contents) == 1, f"FATAL: Extra unexpected files found in ZIP: {contents}"
    test_json = json.loads(zipf.read("prediction.json").decode("utf-8"))
    assert len(test_json) == len(test_queries), "FATAL: Corrupted JSON inside ZIP!"

print("✓ Verified submission.zip integrity.")

# Save evaluation report
val_report = {
    "experiment": EXPERIMENT_NAME,
    "strategy": winning_mode,
    "dev_bm25_f1": float(f1_bm25),
    "dev_tfidf_f1": float(f1_tfidf),
    "dev_best_f1": float(winning_f1),
    "best_alpha": float(best_alpha) if winning_mode == "BLENDED" else None,
    "test_queries_scored": len(predictions)
}
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(val_report, f, indent=2)
print(f"✓ Saved validation report to {METRICS_PATH}")

# Optional non-blocking push to Hugging Face Hub
if HF_TOKEN:
    try:
        api = HfApi(token=HF_TOKEN)
        api.upload_file(
            path_or_fileobj=PREDICTION_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/prediction.json",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"{HUB_SUBFOLDER}/submission.zip",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=METRICS_PATH,
            path_in_repo=f"{HUB_SUBFOLDER}/eval_metrics.json",
            repo_id=HF_DEST_REPO,
            repo_type="model"
        )
        print(f"✓ Successfully synchronized submission files to Hugging Face Hub ({HF_DEST_REPO}/{HUB_SUBFOLDER}).")
    except Exception as e:
        print(f"Notice: Hub sync skipped ({e}). Local submission.zip is fully intact and ready.")


### Summary of Baseline X0 Execution
* **Submission File Ready**: `submission.zip` is located in your output directory (`/kaggle/working/phase7_x0/submission.zip` or `./phase7_x0/submission.zip`).
* **Upload to CodaBench**: Download `submission.zip` from your Kaggle notebook output pane and submit it immediately to establish your initial non-zero benchmark on the leaderboard.
* **Proceed to Prompt 2**: With your baseline secured, proceed to train **R1 (Model 4 Full Fine-Tuning Hierarchical Retriever)** to produce the dense anchor model and distillation teacher!
